

uv add nltk

In [ ]:
import nltk
from nltk.corpus import wordnet as wn

# Download wordnet if you haven't already
nltk.download("wordnet")


In [ ]:
from nltk.corpus import wordnet as wn


def get_lowest_common(syn_string1, syn_string2):
    try:
        syn1 = wn.synset(syn_string1)
        syn2 = wn.synset(syn_string2)
    except Exception:
        return "One of the synset names is invalid."

    # Guard rail: Ensure neither is None
    if syn1 and syn2:
        return syn1.lowest_common_hypernyms(syn2)
    else:
        return "Could not find one or both synsets."

In [ ]:
wagon_synset = wn.synset_from_pos_and_offset("n", 2124072)

In [26]:
diaper_synset1 = wn.synset_from_pos_and_offset("n", 3188531)
diaper_synset1

Synset('diaper.n.01')

In [28]:
from nltk.corpus import wordnet as wn

In [30]:
diaper_synset2 = wn.synset("dog")


ValueError: not enough values to unpack (expected 3, got 1)

In [ ]:
wagon_synset = wn.synset_from_pos_and_offset("n", 2124072)  # Siberian Husky
malamute_synset = wn.synset_from_pos_and_offset("n", 2108112)  # Malamute

In [ ]:
# 1. Find the Lowest Common Ancestor
print(
    "Husky & Malamute Ancestor:", wagon_synset.lowest_common_hypernyms(malamute_synset)
)
# Output: [Synset('dog.n.01')]

print("Husky & Airliner Ancestor:", wagon_synset.lowest_common_hypernyms(diaper_synset))
# Output: [Synset('entity.n.01')] (They only meet at the very top root of reality!)

AttributeError: 'NoneType' object has no attribute 'lowest_common_hypernyms'

In [ ]:
# 2. Get a path-based similarity score (returns a float between 0 and 1)
sim_dog = wagon_synset.path_similarity(malamute_synset)
sim_plane = wagon_synset.path_similarity(diaper_synset)

print(f"Husky-Malamute Similarity: {sim_dog:.4f}")  # High similarity
print(f"Husky-Airliner Similarity: {sim_plane:.4f}")  # Very low similarity

AttributeError: 'NoneType' object has no attribute 'lowest_common_hypernyms'

In [ ]:
import json
import torch
import numpy as np
from tqdm import tqdm
import nltk
from nltk.corpus import wordnet as wn

# Ensure wordnet is downloaded
nltk.download("wordnet")


def get_imagenet_synsets(class_index_json_path):
    """
    Loads standard ImageNet index file and converts IDs to NLTK Synsets.
    File format is typically: {"0": ["n02119789", "pen_point"], ...}
    """
    with open(class_index_json_path, "r") as f:
        class_idx = json.load(f)

    # Sort by class index (0 to 999) to match your model's output alignment
    sorted_indices = sorted(class_idx.keys(), key=int)

    synsets = []
    for idx in sorted_indices:
        wnid = class_idx[idx][0]  # e.g., 'n02119789'
        pos = wnid[0]  # 'n' (noun)
        offset = int(wnid[1:])  # 2119789

        try:
            synset = wn.synset_from_pos_and_offset(pos, offset)
            synsets.append(synset)
        except Exception as e:
            print(f"Warning: Could not parse synset for {wnid}. Using None fallback.")
            synsets.append(None)

    return synsets


def compute_similarity_matrix(synsets):
    K = len(synsets)
    S = np.zeros((K, K))

    # Using tqdm to monitor progress over the 1,000 classes
    print("Computing WordNet Path Similarities...")
    for i in tqdm(range(K)):
        syn_i = synsets[i]
        if syn_i is None:
            continue

        # Initialize diagonal to 0 (since label smoothing only goes to *incorrect* classes)
        S[i, i] = 0.0

        # J starts at i + 1 to only compute the upper triangle (Symmetrical optimization)
        for j in range(i + 1, K):
            syn_j = synsets[j]
            if syn_j is None:
                continue

            # Compute path similarity (returns float between 0 and 1, or None)
            sim = syn_i.path_similarity(syn_j) or 0.0

            # Mirror the results across the diagonal
            S[i, j] = sim
            S[j, i] = sim

    return S


# --- Execution ---
# Replace with the path to your mapping file
# (You can download this JSON from many open-source repos online)
json_path = "imagenet_class_index.json"

# 1. Gather all 1,000 synsets in chronological order
synsets = get_imagenet_synsets(json_path)

# 2. Iterate and compute similarities
raw_similarity_matrix = compute_similarity_matrix(synsets)

# 3. Save it to disk so you never have to calculate it again!
np.save("imagenet_semantic_similarity.npy", raw_similarity_matrix)
print("Finished! Matrix saved successfully.")

[nltk_data] Downloading package wordnet to /home/lucas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


FileNotFoundError: [Errno 2] No such file or directory: 'imagenet_class_index.json'

In [31]:
"""
Semantic Label Smoothing for ImageNet using WordNet (via NLTK)

Standard label smoothing spreads epsilon uniformly over all wrong classes.
Semantic label smoothing instead spreads it according to WordNet semantic
similarity to the true class, so confusing "husky" with "malamute" is
penalized less than confusing it with "toaster".

Usage:
    python semantic_label_smoothing.py --wnids wnids.txt

Where wnids.txt has one ImageNet WordNet ID per line (e.g. n01440764),
in the same order as your model's class indices (index 0 -> line 0, etc).
"""

import argparse
import pickle
from pathlib import Path

import numpy as np
import nltk
from nltk.corpus import wordnet as wn


In [32]:
# ---------------------------------------------------------------------------
# 1. Setup
# ---------------------------------------------------------------------------
def ensure_wordnet():
    for pkg in ["wordnet", "omw-1.4"]:
        try:
            nltk.data.find(f"corpora/{pkg}")
        except LookupError:
            nltk.download(pkg)

In [ ]:
# ---------------------------------------------------------------------------
# 2. Map ImageNet class index -> synset
# ---------------------------------------------------------------------------
def load_imagenet_synsets(wnid_list):
    """
    wnid_list: ordered list of WordNet IDs like 'n01440764', in the same
               order as your model's class indices.
    Returns a list of nltk Synset objects, same order.
    """
    synsets = []
    for wnid in wnid_list:
        pos = wnid[0]  # 'n' for noun; all ImageNet-1k classes are nouns
        offset = int(wnid[1:])
        synsets.append(wn.synset_from_pos_and_offset(pos, offset))
    return synsets


# ---------------------------------------------------------------------------
# 3. Semantic similarity matrix
# ---------------------------------------------------------------------------
def build_similarity_matrix(synsets, metric="wup"):
    """
    metric: 'wup' (Wu-Palmer, tree-depth based, no corpus needed)
            'path' (1 / shortest path length)
    Returns an (N, N) float32 numpy array in [0, 1].
    """
    n = len(synsets)
    sim = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(i, n):
            if i == j:
                s = 1.0
            elif metric == "wup":
                s = synsets[i].wup_similarity(synsets[j]) or 0.0
            elif metric == "path":
                s = synsets[i].path_similarity(synsets[j]) or 0.0
            else:
                raise ValueError(f"unknown metric: {metric}")
            sim[i, j] = sim[j, i] = s
    return sim


# ---------------------------------------------------------------------------
# 4. Similarity -> smoothed target distribution
# ---------------------------------------------------------------------------
def build_smoothing_matrix(sim, epsilon=0.1, temperature=1.0):
    """
    For true class i:
      - class i itself gets (1 - epsilon)
      - epsilon is spread over j != i proportional to softmax(sim[i, j] / T)

    Higher temperature -> flatter (closer to uniform smoothing).
    Lower temperature -> mass concentrated on the most similar classes.

    Returns (N, N) matrix Q, where Q[i] is the target distribution used
    when the ground-truth label is class i.
    """
    n = sim.shape[0]
    Q = np.zeros_like(sim)
    for i in range(n):
        row = sim[i].copy()
        row[i] = -np.inf  # exclude self, handled separately below
        row = row / temperature
        finite = np.isfinite(row)
        row_max = row[finite].max()
        exp_row = np.where(finite, np.exp(row - row_max), 0.0)
        exp_row /= exp_row.sum()
        Q[i] = exp_row * epsilon
        Q[i, i] = 1.0 - epsilon
    return Q


# ---------------------------------------------------------------------------
# 5. PyTorch loss
# ---------------------------------------------------------------------------
def make_semantic_label_smoothing_loss(Q, device="cpu"):
    """
    Q: (num_classes, num_classes) numpy array from build_smoothing_matrix.
    Returns loss_fn(logits, targets) -> scalar loss tensor.
    """
    import torch
    import torch.nn.functional as F

    Q_t = torch.tensor(Q, dtype=torch.float32, device=device)

    def loss_fn(logits, targets):
        # logits: (batch, num_classes), targets: (batch,) int64
        log_probs = F.log_softmax(logits, dim=-1)
        target_dist = Q_t[targets]  # (batch, num_classes)
        return -(target_dist * log_probs).sum(dim=-1).mean()

    return loss_fn


# ---------------------------------------------------------------------------
# CLI: precompute and cache the matrices (this is the slow, one-time step)
# ---------------------------------------------------------------------------


In [34]:
ensure_wordnet()

[nltk_data] Downloading package wordnet to /home/lucas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/lucas/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# get wnids from data/train/
from pathlib import Path

data_path = Path("/home/lucas/code/2026_hackathon/IMAGINE_hackathon_2026_arch/data")
data_dirs = data_path.glob("train/*")
wnids = [dir.name for dir in data_dirs]
wnids.sort()

synsets = load_imagenet_synsets(wnids)
synsets = synsets[899:906]
synsets

[Synset('water_jug.n.01'),
 Synset('water_tower.n.01'),
 Synset('whiskey_jug.n.01'),
 Synset('whistle.n.04'),
 Synset('wig.n.01'),
 Synset('window_screen.n.01'),
 Synset('window_shade.n.01')]

In [ ]:
print(
    "Computing pairwise WordNet similarity (this can take a few minutes for 1000 classes)..."
)
sim = build_similarity_matrix(synsets, metric="wup")

Q = build_smoothing_matrix(sim, epsilon=0.1, temperature=0.1)
Q

Computing pairwise WordNet similarity (this can take a few minutes for 1000 classes)...


array([[0.9       , 0.01297666, 0.07994459, 0.00363433, 0.00105372,
        0.001337  , 0.00105372],
       [0.03928502, 0.9       , 0.03928502, 0.01100242, 0.00319   ,
        0.00404757, 0.00319   ],
       [0.07994459, 0.01297666, 0.9       , 0.00363433, 0.00105372,
        0.001337  , 0.00105372],
       [0.02174657, 0.02174657, 0.02174657, 0.9       , 0.01040841,
        0.01394348, 0.01040841],
       [0.00779445, 0.00779445, 0.00779445, 0.012867  , 0.9       ,
        0.03686627, 0.02688337],
       [0.00522163, 0.00522163, 0.00522163, 0.00910082, 0.01946462,
        0.9       , 0.05576968],
       [0.0046186 , 0.0046186 , 0.0046186 , 0.00762433, 0.01592973,
        0.06259014, 0.9       ]], dtype=float32)

: 

In [86]:
Q.sum(axis=1)

array([1.        , 1.        , 1.        , 0.99999994, 0.99999994,
       1.        , 1.        ], dtype=float32)

In [68]:
1.4899202e-02

0.014899202

In [48]:
Q

array([[8.99999976e-01, 9.24128573e-03, 1.04516288e-02, 1.61694195e-02,
        1.04516288e-02, 8.24593846e-03, 9.24128573e-03, 1.19423559e-02,
        1.04516288e-02, 1.38048353e-02],
       [5.50811354e-04, 8.99999976e-01, 7.69830449e-03, 6.56839926e-03,
        7.69830449e-03, 4.68377862e-03, 5.93733182e-03, 1.02442270e-02,
        5.17157279e-02, 4.90312232e-03],
       [5.36164502e-04, 6.62580458e-03, 8.99999976e-01, 7.83843547e-03,
        8.81703850e-03, 5.11016417e-03, 4.45108786e-02, 1.20911663e-02,
        8.81703850e-03, 5.65331336e-03],
       [1.21330039e-03, 8.26918799e-03, 1.14654005e-02, 8.99999976e-01,
        1.14654005e-02, 6.17271243e-03, 8.26918799e-03, 1.65598392e-02,
        1.14654005e-02, 2.51195766e-02],
       [5.23064635e-04, 6.46391837e-03, 8.60161427e-03, 7.64692156e-03,
        8.99999976e-01, 2.23868079e-02, 6.46391837e-03, 3.37969624e-02,
        8.60161427e-03, 5.51518751e-03],
       [7.58867012e-04, 7.23188370e-03, 9.16740205e-03, 7.57055450e-03,
   